In [3]:
import os, json, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from dotenv import load_dotenv

load_dotenv()
warnings.filterwarnings('ignore')
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
os.makedirs("charts", exist_ok=True)

API_URL = os.getenv('API_URL', 'https://data-charts-api.hexlet.app')
DATE_BEGIN = os.getenv('DATE_BEGIN', '2023-03-01')
DATE_END = os.getenv('DATE_END', '2023-09-01')

# Загрузка данных
try:
    rv = requests.get(f"{API_URL}/visits", params={'begin': DATE_BEGIN, 'end': DATE_END}, verify=False, timeout=30)
    rr = requests.get(f"{API_URL}/registrations", params={'begin': DATE_BEGIN, 'end': DATE_END}, verify=False, timeout=30)
    rv.raise_for_status(); rr.raise_for_status()
    df_visits_api = pd.DataFrame(rv.json())
    df_reg_api = pd.DataFrame(rr.json())
except Exception as e:
    print(f"API Error: {e}"); df_visits_api, df_reg_api = pd.DataFrame(), pd.DataFrame()

try:
    df_ads = pd.read_csv('./ads.csv')
except:
    df_ads = pd.DataFrame()

print(f"✅ Данные: {len(df_visits_api)} визитов, {len(df_reg_api)} рег., {len(df_ads)} рекламы")

✅ Данные: 263459 визитов, 21836 рег., 159 рекламы


In [4]:
if not df_visits_api.empty:
    # Подготовка визитов
    df_v = df_visits_api.copy()
    df_v['datetime'] = pd.to_datetime(df_v['datetime'])
    df_v = df_v[~df_v['user_agent'].str.contains('bot', case=False, na=False)]
    df_v = df_v.sort_values('datetime').drop_duplicates(subset=['visit_id'], keep='last')
    df_v['date_group'] = df_v['datetime'].dt.date
    
    # Подготовка регистраций
    df_r = df_reg_api.copy()
    df_r['datetime'] = pd.to_datetime(df_r['datetime'])
    df_r['date_group'] = df_r['datetime'].dt.date
    
    # === Агрегация для графиков ===
    visits_by_date = df_v.groupby('date_group').size().reset_index(name='visits')
    regs_by_date = df_r.groupby('date_group').size().reset_index(name='registrations')
    visits_by_date_platform = df_v.groupby(['date_group', 'platform']).size().unstack(fill_value=0).reset_index()
    reg_by_date_platform = df_r.groupby(['date_group', 'platform']).size().unstack(fill_value=0).reset_index()
    visits_total = visits_by_date.copy()
    regs_total = regs_by_date.copy()
    
    # === conversion.json ===
    v_agg = df_v.groupby(['date_group', 'platform']).size().reset_index(name='visits')
    r_agg = df_r.groupby(['date_group', 'platform']).size().reset_index(name='registrations')
    df_conv = pd.merge(v_agg, r_agg, on=['date_group', 'platform'], how='outer').fillna(0)
    df_conv['visits'] = df_conv['visits'].astype(int)
    df_conv['registrations'] = df_conv['registrations'].astype(int)
    df_conv['conversion'] = (df_conv['registrations'] / df_conv['visits'] * 100).round(2)
    
    # Сортировка по дате + платформе в порядке ['android','ios','web']
    df_conv['platform'] = pd.Categorical(df_conv['platform'], categories=['android','ios','web'], ordered=True)
    df_conv = df_conv.sort_values(['date_group', 'platform']).reset_index(drop=True)
    df_conv['platform'] = df_conv['platform'].astype(str)
    
    # Сохранение JSON с правильным форматом
    conv_dict = {
        'date_group': (pd.to_datetime(df_conv['date_group']).astype(np.int64) // 10**6).tolist(),
        'platform': df_conv['platform'].tolist(),
        'visits': df_conv['visits'].tolist(),
        'registrations': df_conv['registrations'].tolist(),
        'conversion': [round(float(c), 2) for c in df_conv['conversion'].tolist()]
    }
    with open('./conversion.json', 'w') as f:
        json.dump(conv_dict, f)
    print("✅ conversion.json")
    
    # === ads.json ===
    if not df_ads.empty:
        df_ads['date_group'] = pd.to_datetime(df_ads['date']).dt.date
        ads_agg = df_ads.groupby('date_group').agg({'cost': 'sum', 'utm_campaign': 'first'}).reset_index()
        df_final = pd.merge(visits_by_date, regs_by_date, on='date_group', how='outer').fillna(0)
        df_final = pd.merge(df_final, ads_agg, on='date_group', how='left')
        df_final['visits'] = df_final['visits'].astype(int)
        df_final['registrations'] = df_final['registrations'].astype(int)
        df_final['cost'] = df_final['cost'].fillna(0).astype(int)
        df_final['utm_campaign'] = df_final['utm_campaign'].fillna('none')
        df_final = df_final.sort_values('date_group').reset_index(drop=True)
        df_final[['date_group', 'visits', 'registrations', 'cost', 'utm_campaign']].to_json('./ads.json')
        print("✅ ads.json")
    
    # Подготовка для графиков с рекламой
    ads_by_date = df_ads.groupby('date_group')['utm_campaign'].first().reset_index() if not df_ads.empty else pd.DataFrame()
    print("✅ Данные подготовлены")

✅ conversion.json
✅ ads.json
✅ Данные подготовлены


In [5]:
if not df_visits_api.empty:
    colors = {'android': '#4A90E2', 'ios': '#F5A623', 'web': '#50E3C2'}
    
    # 1. Total Visits
    plt.figure(figsize=(14, 6), dpi=100)
    b = plt.bar(visits_by_date['date_group'], visits_by_date['visits'], color='skyblue', edgecolor='black', linewidth=0.5)
    for bar, val in zip(b, visits_by_date['visits']): plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 15, f'{int(val)}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    plt.title('Total Visits'); plt.xlabel('date_group'); plt.ylabel('visits'); plt.xticks(rotation=45, ha='right'); plt.grid(axis='y', linestyle='--', alpha=0.6)
    plt.tight_layout(); plt.savefig('charts/total_visits.png', dpi=150, bbox_inches='tight'); plt.close()
    
    # 2. Visits by Platform
    plt.figure(figsize=(14, 6), dpi=100)
    plt.bar(visits_by_date_platform['date_group'], visits_by_date_platform.get('android', 0), color=colors['android'], label='android', edgecolor='white')
    plt.bar(visits_by_date_platform['date_group'], visits_by_date_platform.get('ios', 0), bottom=visits_by_date_platform.get('android', 0), color=colors['ios'], label='ios', edgecolor='white')
    plt.bar(visits_by_date_platform['date_group'], visits_by_date_platform.get('web', 0), bottom=visits_by_date_platform.get('android', 0) + visits_by_date_platform.get('ios', 0), color=colors['web'], label='web', edgecolor='white')
    plt.title('Visits by Platform (Stacked)'); plt.legend(title='platform', loc='upper right'); plt.xticks(rotation=45, ha='right'); plt.tight_layout()
    plt.savefig('charts/visits_by_platform.png', dpi=150, bbox_inches='tight'); plt.close()
    
    # 3. Total Registrations
    plt.figure(figsize=(14, 6), dpi=100)
    b = plt.bar(regs_by_date['date_group'], regs_by_date['registrations'], color='skyblue', edgecolor='black', linewidth=0.5)
    for bar, val in zip(b, regs_by_date['registrations']): plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 5, f'{int(val)}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    plt.title('Total Registrations'); plt.xlabel('date_group'); plt.ylabel('registrations'); plt.xticks(rotation=45, ha='right'); plt.tight_layout()
    plt.savefig('charts/total_registrations.png', dpi=150, bbox_inches='tight'); plt.close()
    
    # 4. Registrations by Platform
    plt.figure(figsize=(14, 6), dpi=100)
    plt.bar(reg_by_date_platform['date_group'], reg_by_date_platform.get('android', 0), color=colors['android'], label='android', edgecolor='white')
    plt.bar(reg_by_date_platform['date_group'], reg_by_date_platform.get('ios', 0), bottom=reg_by_date_platform.get('android', 0), color=colors['ios'], label='ios', edgecolor='white')
    plt.bar(reg_by_date_platform['date_group'], reg_by_date_platform.get('web', 0), bottom=reg_by_date_platform.get('android', 0) + reg_by_date_platform.get('ios', 0), color=colors['web'], label='web', edgecolor='white')
    plt.title('Registrations by Platform (Stacked)'); plt.legend(title='platform', loc='upper right'); plt.xticks(rotation=45, ha='right'); plt.tight_layout()
    plt.savefig('charts/registrations_by_platform.png', dpi=150, bbox_inches='tight'); plt.close()
    
    # 5. Average Conversion
    df_ct = pd.merge(visits_by_date, regs_by_date, on='date_group')
    df_ct['conversion'] = (df_ct['registrations'] / df_ct['visits'] * 100).round(0)
    plt.figure(figsize=(12, 6), dpi=100)
    plt.plot(df_ct['date_group'], df_ct['conversion'], color='#4A90E2', marker='o', markersize=6, label='Общая конверсия')
    for i, row in df_ct.iterrows():
        off = 1.5 if i % 2 == 0 else -2
        plt.text(row['date_group'], row['conversion'] + off, f"{int(row['conversion'])}%", ha='center', va='bottom' if off > 0 else 'top', fontsize=9)
    plt.title('Average Conversion'); plt.legend(loc='upper right'); plt.xticks(rotation=45, ha='right'); plt.tight_layout()
    plt.savefig('charts/average_conversion.png', dpi=150, bbox_inches='tight'); plt.close()
    print("✅ Графики 1-5 сохранены")

✅ Графики 1-5 сохранены


In [6]:
if not df_visits_api.empty:
    import matplotlib.colors as mcolors
    
    # 6. Conversion by Platform
    df_cp = pd.merge(df_v.groupby(['date_group', 'platform']).size().reset_index(name='v'), df_r.groupby(['date_group', 'platform']).size().reset_index(name='r'), on=['date_group', 'platform'], how='outer').fillna(0)
    df_cp['conv'] = (df_cp['r'] / df_cp['v'] * 100).round(0)
    fig, axes = plt.subplots(3, 1, figsize=(12, 14), dpi=100, sharex=True)
    for ax, p in zip(axes, ['android', 'ios', 'web']):
        d = df_cp[df_cp['platform'] == p].sort_values('date_group')
        if d.empty: continue
        ax.plot(d['date_group'], d['conv'], color=colors[p], marker='o', label=p)
        for i, row in d.iterrows(): ax.text(row['date_group'], row['conv'] + 2, f"{int(row['conv'])}%", ha='center', fontsize=8)
        ax.set_title(f'Conversion {p}'); ax.legend(loc='upper left'); ax.set_ylim(0, d['conv'].max() * 1.2 + 10)
    plt.xlabel('Date'); plt.tight_layout()
    plt.savefig('charts/conversion_by_platform.png', dpi=150, bbox_inches='tight'); plt.close()
    
    # 7. Ad Costs
    ac = df_ads.groupby('date_group')['cost'].sum().reset_index() if not df_ads.empty else pd.DataFrame(columns=['date_group', 'cost'])
    plt.figure(figsize=(14, 6), dpi=100)
    if not ac.empty:
        plt.plot(ac['date_group'], ac['cost'], color='#4A90E2', marker='o', markersize=5)
        for i, row in ac.iterrows():
            off = 12 if i % 2 == 0 else -25
            plt.text(row['date_group'], row['cost'] + off, f"{int(row['cost'])} RUB", ha='center', va='bottom' if off > 0 else 'top', fontsize=8, fontweight='bold')
    plt.title('Aggregated Ad Campaign Costs (by day)'); plt.xlabel('Date'); plt.ylabel('Cost (RUB)'); plt.xticks(rotation=45, ha='right'); plt.tight_layout()
    plt.savefig('charts/ad_costs.png', dpi=150, bbox_inches='tight'); plt.close()
    
    # 8. Visits with Ads
    vt = visits_total.copy(); vt['date_group'] = pd.to_datetime(vt['date_group'])
    ads_p = df_ads.copy() if not df_ads.empty else pd.DataFrame()
    if not ads_p.empty: ads_p['date_group'] = pd.to_datetime(ads_p['date'])
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(vt['date_group'], vt['visits'], color='black', marker='o', markersize=5, linewidth=1.5, label='Visits')
    ax.axhline(y=vt['visits'].mean(), color='gray', linestyle='--', linewidth=1, label='Average Number of Visits')
    if not ads_p.empty:
        cg = ads_p.groupby(['utm_source', 'utm_medium', 'utm_campaign'])
        tc = list(mcolors.TABLEAU_COLORS.values())
        for i, ((s, m, c), g) in enumerate(cg):
            ax.axvspan(g['date_group'].min(), g['date_group'].max(), color=tc[i % len(tc)], alpha=0.4, label=f"{s} {m} {c}")
    ax.set_title("Visits during marketing active days"); ax.set_ylabel("Unique Visits"); ax.legend(loc='lower left', fontsize=10); fig.autofmt_xdate(rotation=45); plt.tight_layout()
    plt.savefig('charts/visits_with_ads.png', dpi=150, bbox_inches='tight'); plt.close()
    
    # 9. Registrations with Ads
    rt = regs_total.copy(); rt['date_group'] = pd.to_datetime(rt['date_group'])
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(rt['date_group'], rt['registrations'], color='#2E8B57', marker='o', markersize=5, linewidth=1.5, label='Registrations')
    ax.axhline(y=rt['registrations'].mean(), color='gray', linestyle='--', linewidth=1, label='Average Number of Registration')
    if not ads_p.empty:
        tc = list(mcolors.TABLEAU_COLORS.values())
        for i, ((s, m, c), g) in enumerate(cg):
            ax.axvspan(g['date_group'].min(), g['date_group'].max(), color=tc[i % len(tc)], alpha=0.4, label=f"{s} {m} {c}")
    ax.set_title("Registrations during marketing active days"); ax.set_ylabel("Unique Users"); ax.legend(loc='upper right', fontsize=10); fig.autofmt_xdate(rotation=45); plt.tight_layout()
    plt.savefig('charts/registrations_with_ads.png', dpi=150, bbox_inches='tight'); plt.close()
    print("✅ Графики 6-9 сохранены")
    print("🎉 ВСЁ ГОТОВО!")

✅ Графики 6-9 сохранены
🎉 ВСЁ ГОТОВО!
